## A classification model to predict whether a donor will become inactive soon

In [29]:
from IPython.display import display, HTML

def show(df, height=400):
    display(HTML(f'<div style="overflow:auto;max-height:{height}px">{df.to_html()}</div>'))



In [30]:
import pandas as pd

sdf = pd.read_csv('../datasets/supporters.csv')
sdf.info()


<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   supporter_id         60 non-null     int64
 1   supporter_type       60 non-null     str  
 2   display_name         60 non-null     str  
 3   organization_name    4 non-null      str  
 4   first_name           56 non-null     str  
 5   last_name            56 non-null     str  
 6   relationship_type    60 non-null     str  
 7   region               60 non-null     str  
 8   country              60 non-null     str  
 9   email                60 non-null     str  
 10  phone                60 non-null     str  
 11  status               60 non-null     str  
 12  created_at           60 non-null     str  
 13  first_donation_date  59 non-null     str  
 14  acquisition_channel  60 non-null     str  
dtypes: int64(1), str(14)
memory usage: 7.2 KB


In [31]:
sdf[sdf['first_name'].isnull()]

,supporter_id,supporter_type,display_name,organization_name,first_name,last_name,relationship_type,region,country,email,phone,status,created_at,first_donation_date,acquisition_channel
12,13,PartnerOrganization,Hope Group,Hope Group,NaN,NaN,Local,Luzon,Philippines,hope-group@hopegroup.ph,+63 976 726 9666,Active,2022-03-02 00:00:00,2023-02-10,PartnerReferral
37,38,PartnerOrganization,Faith Partners,Faith Partners,NaN,NaN,PartnerOrganization,Mindanao,Philippines,faith-partners@faithpartners.ph,+63 929 536 9976,Active,2022-07-05 00:00:00,2025-10-28,WordOfMouth
45,46,PartnerOrganization,Faith Alliance,Faith Alliance,NaN,NaN,Local,Luzon,Philippines,faith-alliance@faithalliance.ph,+63 906 185 8022,Inactive,2022-08-14 00:00:00,2025-04-22,Event
49,50,PartnerOrganization,Bright Foundation,Bright Foundation,NaN,NaN,Local,Visayas,Philippines,bright-foundation@brightfoundation.ph,+63 935 677 4571,Inactive,2022-09-03 00:00:00,2025-01-24,Event


In [32]:
ddf = pd.read_csv('../datasets/donations.csv')

ddf.head()

,donation_id,supporter_id,donation_type,donation_date,is_recurring,campaign_name,channel_source,currency_code,amount,estimated_value,impact_unit,notes,referral_post_id
0,1,42,Monetary,2025-12-31,False,NaN,Campaign,PHP,717.18,717.18,pesos,In support of safehouse operations,NaN
1,2,25,Time,2025-12-02,True,Year-End Hope,Event,NaN,NaN,35.15,hours,Community outreach support,NaN
2,3,19,Monetary,2024-12-02,False,NaN,PartnerReferral,PHP,1074.65,1074.65,pesos,Campaign support,NaN
3,4,33,Monetary,2023-09-11,False,NaN,PartnerReferral,PHP,1230.56,1230.56,pesos,In support of safehouse operations,NaN
4,5,24,InKind,2023-11-08,True,GivingTuesday,SocialMedia,NaN,NaN,1177.41,items,In support of safehouse operations,421.0


In [33]:
#combine supporters and donations table into a separate df
sdf.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   supporter_id         60 non-null     int64
 1   supporter_type       60 non-null     str  
 2   display_name         60 non-null     str  
 3   organization_name    4 non-null      str  
 4   first_name           56 non-null     str  
 5   last_name            56 non-null     str  
 6   relationship_type    60 non-null     str  
 7   region               60 non-null     str  
 8   country              60 non-null     str  
 9   email                60 non-null     str  
 10  phone                60 non-null     str  
 11  status               60 non-null     str  
 12  created_at           60 non-null     str  
 13  first_donation_date  59 non-null     str  
 14  acquisition_channel  60 non-null     str  
dtypes: int64(1), str(14)
memory usage: 7.2 KB


In [37]:
show(sdf)

,supporter_id,supporter_type,display_name,organization_name,first_name,last_name,relationship_type,region,country,email,phone,status,created_at,first_donation_date,acquisition_channel
0,1,SocialMediaAdvocate,Mila Alvarez,NaN,Mila,Alvarez,Local,Luzon,Philippines,mila-alvarez@smart.com.ph,+63 997 578 1887,Active,2022-01-01 00:00:00,2023-07-02,SocialMedia
1,2,Volunteer,Aria Brown,NaN,Aria,Brown,Local,Mindanao,Philippines,aria-brown@pldt.net.ph,+63 927 354 4139,Active,2022-01-06 00:00:00,2023-09-25,SocialMedia
2,3,MonetaryDonor,Noah Chen,NaN,Noah,Chen,Local,Luzon,Philippines,noah-chen@globe.com.ph,+63 917 553 2604,Active,2022-01-11 00:00:00,2023-06-25,SocialMedia
3,4,MonetaryDonor,Liam Diaz,NaN,Liam,Diaz,PartnerOrganization,Mindanao,Philippines,liam-diaz@globe.com.ph,+63 945 516 8956,Active,2022-01-16 00:00:00,2026-03-01,Church
4,5,InKindDonor,Emma Evans,NaN,Emma,Evans,PartnerOrganization,Mindanao,Philippines,emma-evans@yahoo.com.ph,+63 995 371 8454,Active,2022-01-21 00:00:00,2024-01-18,Website
5,6,MonetaryDonor,Olivia Ford,NaN,Olivia,Ford,International,Visayas,USA,olivia-ford@yahoo.com,+1 (206) 358-4111,Active,2022-01-26 00:00:00,2023-03-22,WordOfMouth
6,7,MonetaryDonor,Ethan Garcia,NaN,Ethan,Garcia,International,Mindanao,USA,ethan-garcia@aol.com,+1 (212) 251-8811,Active,2022-01-31 00:00:00,2023-10-19,Event
7,8,InKindDonor,Isla Hernandez,NaN,Isla,Hernandez,Local,Mindanao,Philippines,isla-hernandez@yahoo.com.ph,+63 953 170 2113,Active,2022-02-05 00:00:00,2023-11-06,Church
8,9,Volunteer,Sophia Ibrahim,NaN,Sophia,Ibrahim,Local,Luzon,Philippines,sophia-ibrahim@globe.com.ph,+63 949 708 1651,Active,2022-02-10 00:00:00,2024-11-04,PartnerReferral
9,10,Volunteer,Lucas Jones,NaN,Lucas,Jones,International,Luzon,Singapore,lucas-jones@gmail.com,+65 8785 6147,Active,2022-02-15 00:00:00,2024-10-18,Event


In [34]:
ddf.info()

<class 'pandas.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   donation_id       420 non-null    int64  
 1   supporter_id      420 non-null    int64  
 2   donation_type     420 non-null    str    
 3   donation_date     420 non-null    str    
 4   is_recurring      420 non-null    bool   
 5   campaign_name     145 non-null    str    
 6   channel_source    420 non-null    str    
 7   currency_code     234 non-null    str    
 8   amount            234 non-null    float64
 9   estimated_value   420 non-null    float64
 10  impact_unit       420 non-null    str    
 11  notes             420 non-null    str    
 12  referral_post_id  77 non-null     float64
dtypes: bool(1), float64(3), int64(2), str(7)
memory usage: 39.9 KB


In [35]:
show(ddf[ddf['campaign_name'].isnull()])

,donation_id,supporter_id,donation_type,donation_date,is_recurring,campaign_name,channel_source,currency_code,amount,estimated_value,impact_unit,notes,referral_post_id
0,1,42,Monetary,2025-12-31,False,NaN,Campaign,PHP,717.18,717.18,pesos,In support of safehouse operations,NaN
2,3,19,Monetary,2024-12-02,False,NaN,PartnerReferral,PHP,1074.65,1074.65,pesos,Campaign support,NaN
3,4,33,Monetary,2023-09-11,False,NaN,PartnerReferral,PHP,1230.56,1230.56,pesos,In support of safehouse operations,NaN
5,6,25,Monetary,2025-09-18,True,NaN,Direct,PHP,678.86,678.86,pesos,Campaign support,NaN
8,9,54,InKind,2023-09-25,True,NaN,Campaign,NaN,NaN,911.84,items,Monthly contribution,NaN
10,11,13,Time,2023-02-10,False,NaN,Event,NaN,NaN,11.39,hours,In support of safehouse operations,NaN
12,13,25,InKind,2025-08-30,True,NaN,Direct,NaN,NaN,433.32,items,Campaign support,NaN
13,14,47,Monetary,2025-10-07,False,NaN,PartnerReferral,PHP,1541.75,1541.75,pesos,In support of safehouse operations,NaN
14,15,1,InKind,2023-07-02,True,NaN,Event,NaN,NaN,300.00,items,Campaign support,NaN
15,16,45,Monetary,2024-10-25,True,NaN,Campaign,PHP,4026.84,4026.84,pesos,Monthly contribution,NaN
